# 02 · Can we trust the teacher inputs and targets?


A frozen encoder can still leak future information through attention or
preprocessing. This stage explains where tokens come from, which tokens the
past input may use, and which person-region features supply the target.
The teaching examples use token arithmetic, not downloaded teacher weights.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Direct gate specification](../../../docs/studies/future-innovation/direct-gate-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation/direct-v2")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Cache the frozen teacher features and check input integrity, teacher repeatability and future isolation. The new direct-v2 protocol performs no person/background replacement or selectivity tests. This requires the prepared cohort and normally one H100. A verified audit rejection finishes with TRAINING BLOCKED and no fitting; missing or corrupt evidence remains an execution error. Completed stages are verified and reused without loading the teacher again.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    device = os.environ.get("FI_NOTEBOOK_DEVICE", "cuda")
    stage_error = attempt_stage("cache-teacher", RUN_ROOT, "--device", device)
    if stage_error is None:
        stage_error = attempt_stage("audit-teacher", RUN_ROOT, "--device", device)

## 1. Count tokens before pooling them

At 384 × 384 pixels, 16-pixel patches form a 24 × 24 grid. Each temporal
token covers two frames. The full clip has 32 temporal groups; the past
input has 16. The actual adapter removes future tokens before attention.
Removing them only after full-clip attention would leave future content
mixed into the retained past features.

In [ ]:
from gavd6_sjepa.research_directions.future_innovation.fi_contracts import FRAME
from gavd6_sjepa.research_directions.future_innovation.fi_token_regions import context_indices, region_masks
past_ids = context_indices()
full_count = FRAME.frames_per_clip // FRAME.tubelet_size * FRAME.grid ** 2
display(pd.DataFrame({"tokens": [len(past_ids), full_count]}, index=["Past before attention", "Full teacher clip"]))
assert past_ids[-1] < FRAME.context_stop_exclusive // FRAME.tubelet_size * FRAME.grid ** 2

## 2. See which spatial region supplies the target

The production region helper expands the person rectangle by one patch
and leaves a further guard band before background patches. Here the box
is invented to show that geometry. Real boxes come from the aligned
annotations and declared crop transformation.

Direct-v2 predicts the person-region feature. Full-clip attention mixes
scene and person information, so this target is contextual. Background
context remains an input when available; an empty background contributes
a zero vector. Prefix-only pixel, optical-flow and token support values
distinguish unavailable measurements from observed zeros in nuisance inputs.
Background-target prediction remains part of the legacy protocol only.

In [ ]:
if MODE == "teach":
    person, background = region_masks(np.array([0.35, 0.15, 0.65, 0.85]))
    regions = np.ones(FRAME.grid ** 2)
    regions[background] = 0
    regions[person] = 2
    from matplotlib.colors import ListedColormap
    fig, ax = plt.subplots(figsize=(5, 4))
    picture = ax.imshow(regions.reshape(FRAME.grid, FRAME.grid),
                        cmap=ListedColormap(["#e7f0f8", "#f7f5ef", "#5f9e7e"]), vmin=0, vmax=2)
    colorbar = fig.colorbar(picture, ax=ax, ticks=[0, 1, 2])
    colorbar.ax.set_yticklabels(["Background", "Guard band", "Person"])
    ax.set(title="Illustrative spatial target regions", xlabel="Patch column", ylabel="Patch row")
    plt.tight_layout(); plt.show()

## 3. Check that inputs and targets can support the comparison

| Saved audit | What must hold |
| --- | --- |
| Repeated teacher inference | The declared numerical tolerance is met |
| Randomized future pixels | The past input features remain unchanged |
| Person-target variance | Nonconstant training features exist |
| Crop and frame alignment | Features correspond to the declared observations |

The numerical thresholds and measurements live in the run artifacts.
Reading a passed flag is inspection of an earlier audit, not a new test.
Direct-v2 checks three frozen windows for repeatability and future
isolation, plus cached person-target variance and artifact integrity.
It does not generate person/background edit contact sheets or require
selectivity thresholds. The legacy run keeps its original tests.
Synthetic fixtures verify software routing, not real teacher behavior.

In [ ]:
if MODE != "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    audit_path = inspection_audit_path(RUN_ROOT)
    if audit_path.is_file():
        audit = read_json(audit_path)
        display(pd.DataFrame(list(audit.get("checks", {}).items()), columns=["saved check", "passed"]))
        failed = [name for name, passed in audit.get("checks", {}).items() if passed is not True]
        print("Failed checks:", ", ".join(failed) if failed else "none")
        if audit_path.name == "validity-summary.json":
            print("Legacy selectivity audit; its rules are preserved for this run.")
            display({key: audit.get(key) for key in
                     ("motion_to_background_change_ratio", "person_edit_direction_fraction")})
        thresholds = RUN_ROOT / "config/thresholds.json"
        if thresholds.is_file():
            display(read_json(thresholds))
        for filename in ("teacher-stability.csv", "causal-leakage.csv", "target-sensitivity.csv"):
            table = read_optional_table(RUN_ROOT, "qc/" + filename)
            if table is not None:
                print(filename)
                display(table)
        print("Saved audit only; inspection does not revalidate it." if MODE == "inspect"
              else "Production commands were attempted; failed checks remain blocking.")
    else:
        print("No saved validity summary. The measurement has not been verified by this notebook.")
    display(artifact_inventory(RUN_ROOT).iloc[3:5])

In [ ]:
if MODE == "execute":
    require_stage_success(stage_error)

## What this step establishes

Token arithmetic explains the interface. Teacher behavior requires the cache and audit stages, which execute mode runs or verifies here. Inspect the saved checks and resolve validity failures before comparing predictors. Fabricated smoke audits remain software tests only.

Continue with [03_matched_predictors_and_controls.ipynb](03_matched_predictors_and_controls.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")